# Portfolio 4 — AquaSense+ NoSQL (Redis) and document model

## 1. Install packages and check files

In [1]:
import os, sys, platform, getpass, subprocess, shutil, re, json
import pandas as pd
#!{sys.executable} -m pip install redis
for d in ("evidence", "document-model"):
    os.makedirs(d, exist_ok=True)
print("Files here:", sorted(os.listdir()))

Files here: ['.ipynb_checkpoints', 'README.md', 'config', 'document-model', 'evidence', 'portfolio4_run.ipynb', 'redis_aquasense.py']


## 2. Connection settings
The password box keeps your PostgreSQL password out of the notebook.

In [3]:
if platform.system() == "Windows":
    os.environ["PATH"] += r";C:\Program Files\PostgreSQL\16\bin;C:\Program Files\Memurai"
os.environ["PGUSER"] = "postgres"
os.environ["PGPASSWORD"] = getpass.getpass("PostgreSQL password: ")
os.environ["PGHOST"] = "localhost"
os.environ["PGDATABASE"] = "aquasense"
os.environ["PGCLIENTENCODING"] = "UTF8"
os.environ["PYTHONUTF8"] = "1"
os.environ["REDIS_HOST"] = "127.0.0.1"
os.environ["REDIS_PORT"] = "6379"

import redis
r = redis.Redis(host="127.0.0.1", port=6379, decode_responses=True)
print("Redis PING ->", r.ping(), "| server version", r.info("server")["redis_version"])
!psql -c "SELECT count(*) AS farms FROM aqua.farm"

PostgreSQL password:  ········


Redis PING -> True | server version 7.2.4
 farms 
-------
   180
(1 row)



## 3. Run the Redis implementation (1–3 minutes)
Loads data from PostgreSQL into Redis and demonstrates 11 patterns:

| | Pattern | Redis feature |
|---|---|---|
| S1 | Load latest readings, time windows, farm locations | Hash, sorted set, geo, pipelining |
| S2 | Pond dashboard lookup vs SQL, and a 32-user load test | `HGETALL` |
| S3 | Last 24 h of oxygen for one pond; 72 h retention | Sorted set `ZRANGEBYSCORE` |
| S4 | Live sensor stream → low-oxygen alerts → write-behind to PostgreSQL | Stream + 2 consumer groups, `MULTI/EXEC`, pub/sub |
| S5 | Offline sync retry is not duplicated | `SET NX` |
| S6 | Phone login codes that expire | TTL |
| S7 | Cooperative FCR leaderboard | Sorted set |
| S8 | Trader in Epe finds farms within 60 km | `GEOSEARCH` |
| S9 | Atomic check-and-decrement of feed stock | Lua script |
| S10 | Cache-aside for the cooperative dashboard | `SET … EX` |
| S11 | Memory and persistence settings | `MEMORY USAGE`, AOF |

Output is saved to `evidence/redis_demo_output.txt`.

In [4]:
run = subprocess.run([sys.executable, "redis_aquasense.py"], capture_output=True, text=True, encoding="utf-8", errors="replace")
print(run.stdout)
print(run.stderr)
open("evidence/redis_demo_output.txt", "w", encoding="utf-8").write(run.stdout)

Connected to 7.2.4 (Memurai) on 127.0.0.1:6379

S1  LOAD from PostgreSQL into Redis structures
Loaded 905 latest-reading hashes, 23,584 time-window entries, 180 geo-indexed farms in 4.83s. DBSIZE = 2171

S2  KEY-VALUE LOOKUP — pond dashboard header (latest reading of pond 316)
redis> HGETALL pond:316:latest
       {'farm_id': '64', 'unit_code': 'TANK-01', 'ts': '2026-11-03T23:00:00+01:00', 'ph': '6.78', 'do': '4.42', 'temp': '27.3', 'nh3': '0.322'}
Single client, median of 2,000 calls: Redis HGETALL = 404 µs | PostgreSQL (indexed) = 354 µs.
       -> With one client on the same machine both are dominated by the network round trip; no real difference.
32 concurrent Python clients x 300 farm-overview screens: Redis = 857 screens/s | PostgreSQL = 1,135 screens/s.
       -> PostgreSQL was faster here. This test is client-bound: redis-py is pure Python (GIL-limited) and the
          Redis version needs two round trips per screen (SMEMBERS, then a pipeline of HGETALLs), while psycopg2
     

8152

## 4. Key results at a glance

In [5]:
keep = ("Connected", "Loaded", "Single client", "screens/s", "old readings", "raised", "XPENDING", "writer-1",
        "call 1", "call 2", "used_memory", "aof_enabled", "farms\n", "[1,", "[0,")
lines = run.stdout.splitlines()
for i, l in enumerate(lines):
    if any(k in l for k in keep) or l.startswith("redis> SET sync") or l.startswith("redis> GET otp") or l.startswith("redis> GEOSEARCH"):
        nxt = lines[i + 1] if i + 1 < len(lines) and l.startswith("redis>") else ""
        print(l[:160]); 
        if nxt: print(nxt[:160])

Connected to 7.2.4 (Memurai) on 127.0.0.1:6379
Loaded 905 latest-reading hashes, 23,584 time-window entries, 180 geo-indexed farms in 4.83s. DBSIZE = 2171
Single client, median of 2,000 calls: Redis HGETALL = 404 µs | PostgreSQL (indexed) = 354 µs.
32 concurrent Python clients x 300 farm-overview screens: Redis = 857 screens/s | PostgreSQL = 1,135 screens/s.
       48 old readings removed, 36 kept
worker-1 processed 12 messages, raised 4 low-oxygen alerts
redis> XPENDING stream:readings alert-workers
       {'pending': 0, 'min': None, 'max': None, 'consumers': []}
writer-1 batch-inserted 12 rows into PostgreSQL -> verified in SQL: (12, Decimal('1.70')) (count, min DO)
redis> SET sync:device-7f3a:rec-000184 done NX EX 86400
       True
redis> SET sync:device-7f3a:rec-000184 done NX EX 86400   (retry of the same record)
       None
redis> GET otp:+2348030015838   (after 2.2 s, code was set with EX 2)
       None
redis> GEOSEARCH farms:geo FROMLONLAT 3.983 6.583 BYRADIUS 60 km ASC WITHDIS

## 5. The key namespace actually in Redis
Digits replaced by `N`, so you see the key *patterns* and how many of each. Saved to `evidence/redis_key_patterns.txt`.

In [6]:
from collections import Counter
patterns = Counter(re.sub(r"\d+", "N", k) for k in r.scan_iter("*", count=1000))
text = "\n".join(f"{n:6}  {p}  ({r.type(next(r.scan_iter(p.replace('N', '*'), count=1000)))})" for p, n in patterns.most_common())
open("evidence/redis_key_patterns.txt", "w", encoding="utf-8").write(text)
print(text)

   905  pond:N:do  (zset)
   905  pond:N:latest  (hash)
   180  farm:N  (hash)
   180  farm:N:ponds  (set)
    15  coop:N:fcr  (zset)
     1  otp:+N  (string)
     1  harvest:ready  (set)
     1  farms:geo  (zset)
     1  farm:N:feedstock  (hash)
     1  farm:N:alerts  (zset)
     1  cache:dash:LA:N  (string)
     1  sync:device-NfNa:rec-N  (string)
     1  stream:readings  (stream)


## 6. Fair server benchmark: Redis/Memurai benchmark tool vs pgbench
Both are C clients, 32 connections, so this compares the servers rather than Python. Saved to `evidence/server_benchmark_output.txt`.

In [7]:
bench = shutil.which("memurai-benchmark") or shutil.which("redis-benchmark")
out = []
def run_cmd(title, cmd):
    res = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8", errors="replace")
    text = (res.stdout + res.stderr).replace("\r", "\n")
    keep = [l.strip() for l in text.splitlines() if re.search(r"requests per second|tps|latency average|clients", l)]
    out.append(f"### {title}"); out.extend(keep[-3:])

if bench:
    run_cmd("Redis-compatible server: HGETALL pond:316:latest, 32 clients, 200k requests", [bench, "-q", "-c", "32", "-n", "200000", "HGETALL", "pond:316:latest"])
    run_cmd("Same, pipelined x8 (8 ponds per farm screen)", [bench, "-q", "-c", "32", "-n", "200000", "-P", "8", "HGETALL", "pond:316:latest"])
else:
    out.append("### benchmark tool not found on PATH - skipped")
run_cmd("pgbench: indexed latest-reading query, 32 clients, 15 s, prepared statements",
        ["pgbench", "-n", "-c", "32", "-j", "4", "-T", "15", "-M", "prepared", "-f", "config/pgbench_latest_reading.sql", "aquasense"])
open("evidence/server_benchmark_output.txt", "w", encoding="utf-8").write("\n".join(out))
print("\n".join(out))

### benchmark tool not found on PATH - skipped
### pgbench: indexed latest-reading query, 32 clients, 15 s, prepared statements
number of clients: 32
latency average = 2.278 ms
tps = 14048.179132 (without initial connection time)


## 7. Document model: one production cycle as a MongoDB-style document
Generated from the real PostgreSQL data with `jsonb_build_object`: farm and pond embedded, daily records as an array, samples, harvest and summary. Saved to `document-model/example_batch_document.json`.

In [8]:
import psycopg2
pg = psycopg2.connect(dbname="aquasense")
cur = pg.cursor()
cur.execute("""SELECT jsonb_pretty(jsonb_build_object(
  '_id', 'batch_' || b.batch_id,
  'farm', jsonb_build_object('farm_id', f.farm_id, 'name', f.farm_name, 'state', f.state_code, 'lga', f.lga,
          'location', jsonb_build_object('type','Point','coordinates', jsonb_build_array(f.longitude, f.latitude))),
  'unit', jsonb_build_object('unit_id', cu.unit_id, 'code', cu.unit_code, 'type', cu.unit_type, 'volume_m3', cu.volume_m3),
  'species', s.common_name, 'stocked_on', b.stocked_on, 'closed_on', b.closed_on, 'status', b.status,
  'stocking', jsonb_build_object('count', b.initial_count, 'avg_weight_g', b.initial_avg_weight_g, 'hatchery', b.hatchery_source),
  'daily', (SELECT jsonb_agg(d) FROM (
      SELECT jsonb_build_object('date', fl.fed_on, 'feed_kg', fl.qty_kg, 'feed', fp.brand || ' ' || fp.pellet_size_mm || 'mm',
             'dead', (SELECT COALESCE(SUM(dead_count),0) FROM aqua.mortality_log m WHERE m.batch_id=b.batch_id AND m.recorded_on=fl.fed_on)) d
      FROM aqua.feeding_log fl JOIN aqua.feed_product fp USING (feed_id) WHERE fl.batch_id=b.batch_id ORDER BY fl.fed_on LIMIT 3) x),
  'daily_note', 'first 3 of ' || (SELECT count(*) FROM aqua.feeding_log WHERE batch_id=b.batch_id) || ' daily entries shown',
  'samples', (SELECT jsonb_agg(jsonb_build_object('date', sampled_on, 'avg_weight_g', avg_weight_g) ORDER BY sampled_on) FROM aqua.growth_sample g WHERE g.batch_id=b.batch_id),
  'harvest', (SELECT jsonb_build_object('date', harvested_on, 'fish_count', fish_count, 'total_kg', total_weight_kg) FROM aqua.harvest h WHERE h.batch_id=b.batch_id ORDER BY harvest_id LIMIT 1),
  'summary', jsonb_build_object('feed_kg', v.total_feed_kg, 'dead', v.total_dead, 'survival_pct', v.survival_pct)))
FROM aqua.stock_batch b JOIN aqua.culture_unit cu USING (unit_id) JOIN aqua.farm f USING (farm_id)
JOIN aqua.species s USING (species_id) JOIN aqua.v_batch_status v USING (batch_id)
WHERE cu.farm_id = 64 AND b.status = 'harvested' ORDER BY b.batch_id LIMIT 1""")
doc = cur.fetchone()[0]
open("document-model/example_batch_document.json", "w", encoding="utf-8").write(doc)
print(doc)

{
    "_id": "batch_316",
    "farm": {
        "lga": "Warri South",
        "name": "Nwosu Fish Farm",
        "state": "DE",
        "farm_id": 64,
        "location": {
            "type": "Point",
            "coordinates": [
                5.79008,
                5.43551
            ]
        }
    },
    "unit": {
        "code": "TANK-01",
        "type": "concrete_tank",
        "unit_id": 316,
        "volume_m3": 23.10
    },
    "daily": [
        {
            "date": "2024-12-29",
            "dead": 0,
            "feed": "Coppens 2.0mm",
            "feed_kg": 0.36
        },
        {
            "date": "2024-12-30",
            "dead": 0,
            "feed": "Coppens 2.0mm",
            "feed_kg": 0.36
        },
        {
            "date": "2024-12-31",
            "dead": 0,
            "feed": "Coppens 2.0mm",
            "feed_kg": 0.37
        }
    ],
    "status": "harvested",
    "harvest": {
        "date": "2025-05-30",
        "total_kg": 640.94,
     

## 8. Numbers to copy into the README

In [9]:
t = run.stdout
bm = open("evidence/server_benchmark_output.txt", encoding="utf-8").read()
def find(pat, src=t, g=1):
    m = re.search(pat, src); return m.group(g) if m else "?"
rps = re.findall(r"([0-9.,]+) requests per second", bm)
pd.DataFrame([
    {"Item": "Server", "Value": find(r"Connected to (.+?) on")},
    {"Item": "S1 load (hashes / window entries / geo farms / seconds)", "Value": " / ".join(re.search(r"Loaded (\d+) latest-reading hashes, ([0-9,]+) time-window entries, (\d+) geo-indexed farms in ([0-9.]+)s", t).groups()) if "Loaded" in t else "?"},
    {"Item": "S2 single lookup: Redis µs", "Value": find(r"HGETALL = (\d+) µs")},
    {"Item": "S2 single lookup: PostgreSQL µs", "Value": find(r"PostgreSQL \(indexed\) = (\d+) µs")},
    {"Item": "S2 32 users: Redis screens/s", "Value": find(r"Redis = ([0-9,]+) screens/s")},
    {"Item": "S2 32 users: PostgreSQL screens/s", "Value": find(r"PostgreSQL = ([0-9,]+) screens/s")},
    {"Item": "S3 readings removed / kept", "Value": find(r"(\d+ old readings removed, \d+ kept)")},
    {"Item": "S4 alerts raised", "Value": find(r"raised (\d+) low-oxygen")},
    {"Item": "S8 farms within 60 km", "Value": find(r"BYRADIUS 60 km ASC WITHDIST\n\s+(\d+) farms")},
    {"Item": "S10 cache miss / hit (ms)", "Value": find(r"call 1: cache MISS\s+([0-9.]+)") + " / " + find(r"call 2: cache HIT\s+([0-9.]+)")},
    {"Item": "S11 memory used", "Value": find(r"used_memory_human = (\S+),")},
    {"Item": "Benchmark: Redis req/s / pipelined req/s", "Value": " / ".join(rps) if rps else "?"},
    {"Item": "Benchmark: pgbench tps", "Value": find(r"tps = ([0-9.]+)", bm)},
])

,Item,Value
0,Server,7.2.4 (Memurai)
1,S1 load (hashes / window entries / geo farms /...,"905 / 23,584 / 180 / 4.83"
2,S2 single lookup: Redis µs,404
3,S2 single lookup: PostgreSQL µs,354
4,S2 32 users: Redis screens/s,857
5,S2 32 users: PostgreSQL screens/s,"1,135"
6,S3 readings removed / kept,"48 old readings removed, 36 kept"
7,S4 alerts raised,4
8,S8 farms within 60 km,23
9,S10 cache miss / hit (ms),42.79 / 0.56
